# 3. Temporal queries and authorization reasoning

This notebook demonstrates the existing Semantica-backed path in `northstar_authz/semantica_pipeline.py`:

1. Build the knowledge graph.
2. Take a point-in-time snapshot with `TemporalGraphQuery`.
3. Use `DatalogReasoner` for recursive team and project reachability.
4. Use `Reasoner` for authorization rules.
5. Use `ExplanationGenerator` for an allowed decision.

## Create a point-in-time graph snapshot

In [ ]:
from datetime import datetime, timezone
from northstar_authz.service import load_data, build_graph, temporal_snapshot

data = load_data()
graph = build_graph(data)

as_of = datetime(2026, 11, 1, 12, 0, tzinfo=timezone.utc)
snapshot = temporal_snapshot(graph, as_of)

print("Relationships in snapshot:", snapshot.get("num_relationships"))

## Check the temporal share example

Ethan's direct share to DOC-107 ends at 2026-10-28T00:00:00Z. The interval is half-open: the share is active before the end time, but not at the end time.

In [ ]:
before_expiry = temporal_snapshot(
    graph, datetime(2026, 10, 27, 12, tzinfo=timezone.utc)
)
at_expiry = temporal_snapshot(
    graph, datetime(2026, 10, 28, 0, tzinfo=timezone.utc)
)

def has_ethan_share(result):
    return any(
        edge.get("source") == "Ethan"
        and edge.get("target") == "DOC-107"
        and edge.get("type") == "directShareTo"
        for edge in result.get("relationships", [])
    )

print("Active before expiry:", has_ethan_share(before_expiry))
print("Active at expiry:", has_ethan_share(at_expiry))

## Run authorization decisions through Semantica

In [ ]:
from northstar_authz.semantica_pipeline import evaluate_with_semantica

decision = evaluate_with_semantica(
    data,
    person_id="Alice",
    document_id="DOC-101",
    as_of=as_of,
)
print("Allowed:", decision.allowed)
print("Explanation:", decision.explanation)

## Compare an allowed and a restricted document

In [ ]:
for document_id in ["DOC-101", "DOC-102"]:
    result = evaluate_with_semantica(
        data, "Alice", document_id, as_of
    )
    print(document_id, "->", "ALLOW" if result.allowed else "DENY")
    print(result.explanation)
    print()

## Where the reasoning classes are used

Open `northstar_authz/semantica_pipeline.py` to see the complete implementation:

- `DatalogReasoner` derives recursive team ancestry and project access.
- `Reasoner` applies the IF-THEN authorization rules.
- `ExplanationGenerator` creates an explanation for a matched allow inference.

For the denial path, this POC assembles the explanation in application code. It does not claim that `ExplanationGenerator` explains denials.

**Important:** this is a learning POC, not a production authorization service.